# SentinelAI — RWF-2000 research baseline
Train one visual fight/nonfight component. This does **not** train Urdu speech, bribery context, gestures, weapons or a validated misconduct classifier.

1. Create a **private** Kaggle notebook, import this file, enable a GPU, and attach either your original RWF-2000 source or the [new collection](https://www.kaggle.com/datasets/magicearth25/video-violence-detection-dataset). The new collection includes RLVS and RWF-2000; select its **RWF-2000 subset** for this baseline. Do not combine duplicate copies.
2. Check the dataset's access/use terms. The [original authors](https://github.com/mchengny/RWF2000-Video-Database-for-Violence-Detection) currently withhold downloads for privacy and restrict redistribution/commercial use. A mirror's availability does not settle permission. No dataset is downloaded or republished by this notebook.
3. Enable Internet for the initial torchvision Kinetics-400 pretrained weight download, or use Kaggle's already cached weights. Missing weights stop training.
4. Set `AUTHORIZED_DATASET = True` only after confirming authorized use. Kaggle reports the new collection's license as **Unknown**; do not infer permission from its presence on Kaggle. Adjust the dataset path if automatic discovery finds multiple RWF copies.
5. Run all cells. Download `sentinel-rwf-r3d18-v1.zip` from Output. Keep the notebook private and do not publish source footage.

The supplied `val/` (or `test/`) is held out. A development subset comes only from `train/`. Preparation keeps one copy of identical videos with the same label within each original split, without modifying the source dataset. Every exclusion is recorded in `curation.json`. Exact cross-split duplicates keep their held-out copy; all matching training copies are excluded before development splitting. Conflicting labels or source groups still stop preparation. Results describe this curated subset, not the unchanged published benchmark. Optional source/session groups prevent group leakage; otherwise splitting is clip-level and exact hashes catch only exact duplicates. Do not report these results as local police/mobile accuracy. XD-Violence and a consented Pakistani dataset remain separate future work.


In [ ]:
from pathlib import Path
import sys, json, subprocess, importlib
SOURCE = Path('/kaggle/working/sentinel_src')
FILES = {'sentinel_ml/__init__.py': '"""Shared training/inference contract; no model weights or automatic downloads."""\n', 'sentinel_ml/video.py': '"""Exactly the same RGB sampling and preprocessing in Kaggle and the server."""\nfrom pathlib import Path\nimport hashlib\nimport cv2\nimport numpy as np\n\nPREPROCESS = {"frames": 16, "window_seconds": 5.0, "resize": [128, 171],\n              "crop": 112, "mean": [0.43216, 0.394666, 0.37645],\n              "std": [0.22803, 0.22145, 0.216989], "sampling": "uniform-v1"}\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef video_info(path):\n    cap = cv2.VideoCapture(str(path), cv2.CAP_FFMPEG,\n                           [cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, 10000, cv2.CAP_PROP_READ_TIMEOUT_MSEC, 10000])\n    try:\n        fps, count = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n        if not np.isfinite(fps) or fps <= 0 or count < 1:\n            raise ValueError("Video has no usable frame timing.")\n        return fps, count\n    finally:\n        cap.release()\n\n\ndef windows(duration):\n    if not np.isfinite(duration) or not 0 < duration <= 120.1:\n        raise ValueError("Analysis supports videos up to 120 seconds.")\n    return [(float(start), min(float(start + 5), duration)) for start in range(0, int(np.ceil(duration)), 5)]\n\n\ndef read_clip(path, start=0.0, end=None):\n    fps, count = video_info(path)\n    duration = count / fps\n    end = min(duration, start + 5.0) if end is None else min(end, duration)\n    if not np.isfinite(start) or not np.isfinite(end) or not 0 <= start < end:\n        raise ValueError("Invalid clip interval.")\n    first, last = min(count - 1, int(start * fps)), min(count - 1, max(0, int(np.ceil(end * fps)) - 1))\n    indices = np.linspace(first, last, PREPROCESS["frames"]).round().astype(int)\n    cap, frames = cv2.VideoCapture(str(path), cv2.CAP_FFMPEG,\n                                  [cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, 10000, cv2.CAP_PROP_READ_TIMEOUT_MSEC, 10000]), []\n    try:\n        previous, frame = -1, None\n        for index in indices:\n            if index != previous:\n                cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))\n                ok, frame = cap.read()\n                if not ok:\n                    raise ValueError("A sampled video frame could not be decoded.")\n                previous = index\n            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n            rgb = cv2.resize(rgb, (171, 128), interpolation=cv2.INTER_LINEAR)\n            frames.append(rgb[8:120, 29:141])\n    finally:\n        cap.release()\n    values = np.stack(frames).astype(np.float32) / 255.0\n    values = (values - np.array(PREPROCESS["mean"], np.float32)) / np.array(PREPROCESS["std"], np.float32)\n    return np.ascontiguousarray(values.transpose(3, 0, 1, 2))\n', 'training/__init__.py': '"""Reproducible research training, separate from application data."""\n', 'training/data.py': '"""RWF-style inventory, deterministic development split and measured binary metrics."""\nfrom pathlib import Path\nimport json\nimport random\nfrom sentinel_ml.video import sha256, video_info\n\nEXTENSIONS = {".avi", ".mp4", ".mov", ".mkv", ".webm"}\n\n\ndef dataset_roots(path):\n    path = Path(path)\n    candidates = [path, *sorted(p.parent for p in path.rglob("train") if p.is_dir())]\n    matches = []\n    for candidate in candidates:\n        children = {p.name.casefold(): p for p in candidate.iterdir() if p.is_dir()}\n        if "train" in children and ("val" in children or "test" in children):\n            if candidate not in matches:\n                matches.append(candidate)\n    return matches\n\n\ndef find_root(path):\n    matches = dataset_roots(path)\n    if len(matches) != 1:\n        raise ValueError("Set DATASET_ROOT to exactly one dataset with train/ and val/ (or test/) folders. Collections must select a subset; do not mix RWF copies.")\n    return matches[0]\n\n\ndef inventory(root, seed=42, groups_file=None):\n    root = find_root(root)\n    groups = json.loads(Path(groups_file).read_text()) if groups_file else {}\n    children = {p.name.casefold(): p for p in root.iterdir() if p.is_dir()}\n    if "test" in children and "val" in children:\n        raise ValueError("Both val and test exist. Use an explicit curated RWF two-split root; do not guess the holdout.")\n    source_splits = {"train": children["train"], "test": children.get("test", children.get("val"))}\n    rows, hashes, assigned_groups = [], {}, {}\n    for split, folder in source_splits.items():\n        for video in sorted(folder.rglob("*")):\n            if video.suffix.casefold() not in EXTENSIONS:\n                continue\n            category = video.parent.name.casefold().replace("-", "").replace("_", "")\n            if category not in ("fight", "nonfight"):\n                raise ValueError(f"Unrecognized class folder: {video.parent.name}")\n            relative = video.relative_to(root).as_posix()\n            digest = sha256(video)\n            if digest in hashes:\n                raise ValueError(f"Duplicate content: {relative} and {hashes[digest]}. Curate duplicates before training.")\n            hashes[digest] = relative\n            fps, count = video_info(video)\n            if not .2 <= count/fps <= 5.5:\n                raise ValueError(f"Expected short RWF clips (0.2–5.5 s): {relative}. Do not silently truncate another dataset.")\n            if groups_file and relative not in groups:\n                raise ValueError(f"Missing source/session group for {relative}")\n            group = groups.get(relative, relative)\n            if not isinstance(group, str) or not group:\n                raise ValueError("Group IDs must be nonempty strings.")\n            if group in assigned_groups and assigned_groups[group] != split:\n                raise ValueError("A source/session group crosses the supplied train/holdout boundary.")\n            assigned_groups[group] = split\n            rows.append({"path": str(video), "relative": relative, "label": int(category == "fight"),\n                         "sha256": digest, "group": group, "split": split, "duration": count/fps})\n    training_groups = sorted({r["group"] for r in rows if r["split"] == "train"})\n    rng = random.Random(seed)\n    # Shuffle groups until both classes exist in train and dev, without splitting a group.\n    selected = None\n    for _ in range(1000):\n        rng.shuffle(training_groups)\n        dev = set(training_groups[:max(2, round(len(training_groups)*.2))])\n        if all({r["label"] for r in rows if r["split"] == "train" and ((r["group"] in dev) == is_dev)} == {0, 1} for is_dev in (True, False)):\n            selected = dev\n            break\n    if selected is None or {r["label"] for r in rows if r["split"] == "test"} != {0, 1}:\n        raise ValueError("Each of train, development and held-out test needs both classes and sufficient independent groups.")\n    for row in rows:\n        if row["split"] == "train" and row["group"] in selected:\n            row["split"] = "dev"\n    return rows\n\n\ndef metrics(labels, scores, threshold):\n    if len(labels) != len(scores) or not labels:\n        raise ValueError("Metrics require aligned nonempty labels and scores.")\n    tp = sum(y == 1 and p >= threshold for y, p in zip(labels, scores))\n    tn = sum(y == 0 and p < threshold for y, p in zip(labels, scores))\n    fp = sum(y == 0 and p >= threshold for y, p in zip(labels, scores))\n    fn = sum(y == 1 and p < threshold for y, p in zip(labels, scores))\n    return {"count": len(labels), "threshold": threshold, "confusion_matrix": [[tn, fp], [fn, tp]],\n            "accuracy": (tp+tn)/len(labels), "precision": tp/max(1, tp+fp),\n            "recall": tp/max(1, tp+fn), "f1": 2*tp/max(1, 2*tp+fp+fn)}\n\n\ndef choose_threshold(labels, scores):\n    # Select once on development only. Conservative tie-break: higher threshold.\n    return max((metrics(labels, scores, n/100) for n in range(5, 100)), key=lambda m: (m["f1"], m["threshold"]))\n', 'training/curation.py': '"""Build a linked dataset view; keep originals and a record of every exclusion."""\nimport json\nimport os\nfrom pathlib import Path\nfrom sentinel_ml.video import sha256\nfrom training.data import EXTENSIONS, find_root\n\n\ndef curate(root, destination, groups_file=None, overlap_policy="reject"):\n    if overlap_policy not in ("reject", "keep_holdout"):\n        raise ValueError("Choose reject or keep_holdout for exact train/holdout overlap.")\n    root, destination = find_root(root).resolve(), Path(destination).resolve()\n    if destination == root or destination.is_relative_to(root):\n        raise ValueError("The curated view must be outside the source dataset.")\n    if destination.exists() and any(destination.iterdir()):\n        raise ValueError("Use an empty destination; existing experiments are never overwritten.")\n    folders = {p.name.casefold(): p for p in root.iterdir() if p.is_dir()}\n    if "val" in folders and "test" in folders:\n        raise ValueError("Both val and test exist. Select an explicit two-split dataset first.")\n    groups = json.loads(Path(groups_file).read_text()) if groups_file else {}\n    seen, group_splits, kept, duplicates = {}, {}, [], []\n    print(f"Checking exact duplicates in {root} ...", flush=True)\n    splits = [("train", folders["train"]), ("test", folders.get("test", folders.get("val")))]\n    if overlap_policy == "keep_holdout":\n        # Holdout is inventoried first. Every matching training copy is excluded\n        # before development splitting; no held-out video is moved into training.\n        splits.reverse()\n    for split, folder in splits:\n        for video in sorted(folder.rglob("*")):\n            if video.suffix.casefold() not in EXTENSIONS:\n                continue\n            relative = video.relative_to(root).as_posix()\n            label = video.parent.name.casefold().replace("-", "").replace("_", "")\n            if label not in ("fight", "nonfight"):\n                raise ValueError(f"Unknown class folder: {relative}")\n            if groups_file and relative not in groups:\n                raise ValueError(f"Missing source/session group: {relative}")\n            group = groups.get(relative, relative)\n            if not isinstance(group, str) or not group:\n                raise ValueError("Source/session groups must be nonempty strings.")\n            if group in group_splits and group_splits[group] != split:\n                raise ValueError("A source/session group crosses train and held-out data.")\n            group_splits[group] = split\n            digest = sha256(video)\n            prior = seen.get(digest)\n            if prior:\n                if prior["label"] != label:\n                    raise ValueError(f"Conflicting labels for identical videos: {prior[\'relative\']} and {relative}.")\n                cross_split = prior["split"] != split\n                if cross_split and overlap_policy == "reject":\n                    raise ValueError(f"Duplicate content crosses train/holdout boundary: {prior[\'relative\']} and {relative}. Curate this overlap before training.")\n                if groups_file and prior["group"] != group:\n                    raise ValueError(f"Identical videos have conflicting source/session groups: {relative}.")\n                duplicates.append({"kept": prior["relative"], "excluded": relative, "sha256": digest,\n                                   "source_split": split, "label": label,\n                                   "reason": "train_holdout_overlap" if cross_split else "same_split_duplicate"})\n            else:\n                row = {"relative": relative, "sha256": digest, "split": split, "label": label, "group": group}\n                seen[digest] = row\n                kept.append(row)\n            checked = len(kept) + len(duplicates)\n            if checked % 250 == 0:\n                print(f"Checked {checked} videos; {len(duplicates)} redundant copies found.", flush=True)\n    if not kept:\n        raise ValueError("No supported video files found.")\n    # Validate all hashes/labels/groups before creating any view. Do not edit,\n    # delete, move or copy source videos. Kaggle supports links to /kaggle/input.\n    destination.mkdir(parents=True, exist_ok=True)\n    for row in kept:\n        target = destination/row["relative"]\n        target.parent.mkdir(parents=True, exist_ok=True)\n        try:\n            target.symlink_to(root/row["relative"])\n        except OSError:\n            # Windows local tests can use same-volume hard links without symlink privileges.\n            os.link(root/row["relative"], target)\n    report = {"schema": 2, "policy": "holdout-priority-exact-dedup-v2" if overlap_policy == "keep_holdout" else "same-split-exact-dedup-v2", "source_root": str(root),\n              "source_count": len(kept)+len(duplicates), "kept_count": len(kept),\n              "excluded_count": len(duplicates), "excluded": duplicates,\n              "training_overlap_exclusions": sum(r["reason"] == "train_holdout_overlap" for r in duplicates),\n              "kept_counts": {split: sum(r["split"] == split for r in kept) for split in ("train", "test")},\n              "limitations": "Exact byte duplicates only; reencoded or related footage still needs source-group review."}\n    (destination/"curation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(f"Curation complete: {len(kept)} unique videos; {len(duplicates)} redundant copies excluded. Originals preserved.", flush=True)\n    print(f"Training copies excluded to protect holdout: {report[\'training_overlap_exclusions\']}. Report results as a curated subset, not the unchanged benchmark.", flush=True)\n    print(f"Curation report: {destination/\'curation.json\'}", flush=True)\n    return destination\n', 'training/train_violence.py': '"""Kaggle GPU baseline: R3D-18 transfer learning; no claims of police-domain accuracy."""\nimport argparse\nfrom datetime import datetime, timezone\nimport json\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\nimport numpy as np\nimport torch\nimport torchvision\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.models.video import R3D_18_Weights, r3d_18\nfrom sentinel_ml.video import PREPROCESS, read_clip, sha256\nfrom training.data import inventory, metrics, choose_threshold\n\n\nclass Clips(Dataset):\n    def __init__(self, rows, augment=False):\n        self.rows, self.augment = rows, augment\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        # Both training and inference cover a maximum five-second window uniformly.\n        value = read_clip(row["path"], 0, min(5, row["duration"]))\n        if self.augment and random.random() < .5:\n            value = value[:, :, :, ::-1].copy()\n        return torch.from_numpy(value), row["label"]\n\n\ndef predict(model, loader, device):\n    model.eval()\n    labels, scores = [], []\n    with torch.inference_mode():\n        for x, y in loader:\n            out = model(x.to(device)).softmax(1)[:, 1]\n            if not torch.isfinite(out).all():\n                raise ValueError("Nonfinite model output; no export written.")\n            labels.extend(y.tolist())\n            scores.extend(out.cpu().tolist())\n    return labels, scores\n\n\ndef train(args):\n    if not torch.cuda.is_available() and not args.smoke:\n        raise RuntimeError("Enable a Kaggle GPU accelerator before training. CPU is allowed only for --smoke plumbing tests.")\n    if not args.authorized_dataset and not args.smoke:\n        raise ValueError("Confirm authorized dataset use with --authorized-dataset after reviewing the dataset terms.")\n    if args.epochs < 1 or args.batch_size < 1 or args.workers < 0:\n        raise ValueError("Epochs/batch size must be positive; workers cannot be negative.")\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    torch.cuda.manual_seed_all(args.seed)\n    torch.set_num_threads(2)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    if any(output.iterdir()):\n        raise ValueError("Output folder must be empty; preserve earlier experiment results.")\n    rows = inventory(args.dataset, args.seed, args.groups)\n    print(json.dumps({"inventory_complete": True, "counts": {s: sum(r[\'split\'] == s for r in rows) for s in (\'train\', \'dev\', \'test\')}, "grouped_by_source": bool(args.groups)}), flush=True)\n    public_rows = [{k:v for k,v in r.items() if k != "path"} for r in rows]\n    (output/"split.json").write_text(json.dumps(public_rows, indent=2), encoding="utf-8")\n    curation_file = Path(args.dataset)/"curation.json"\n    if curation_file.is_file():\n        shutil.copyfile(curation_file, output/"curation.json")\n    loaders = {split: DataLoader(Clips([r for r in rows if r["split"] == split], split == "train"),\n                                batch_size=args.batch_size, shuffle=split == "train", num_workers=args.workers,\n                                pin_memory=device.type == "cuda") for split in ("train", "dev", "test")}\n    # Failure to fetch pretrained weights is an error, never silent random initialization.\n    model = r3d_18(weights=None if args.smoke else R3D_18_Weights.KINETICS400_V1)\n    model.fc = torch.nn.Linear(model.fc.in_features, 2)\n    model.to(device)\n    for name, parameter in model.named_parameters():\n        parameter.requires_grad = name.startswith(("layer4.", "fc."))\n    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=args.lr, weight_decay=.01)\n    criterion = torch.nn.CrossEntropyLoss()\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    best, history, start = -1.0, [], time.perf_counter()\n    for epoch in range(args.epochs):\n        # Frozen backbone and batchnorm statistics stay in eval mode for small batches.\n        model.eval()\n        loss_sum, examples = 0.0, 0\n        for x, y in loaders["train"]:\n            x, y = x.to(device), y.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                loss = criterion(model(x), y)\n            if not torch.isfinite(loss):\n                raise ValueError("Nonfinite training loss; inspect the data and lower the learning rate.")\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            loss_sum += loss.item()*len(y)\n            examples += len(y)\n            if args.smoke:\n                break\n        labels, scores = predict(model, loaders["dev"], device)\n        dev_result = metrics(labels, scores, .5)\n        record = {"epoch": epoch+1, "train_loss": loss_sum/examples, "dev": dev_result}\n        history.append(record)\n        print(json.dumps(record), flush=True)\n        if dev_result["f1"] > best:\n            best = dev_result["f1"]\n            torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()}, output/"weights.pt")\n        (output/"history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n    model.load_state_dict(torch.load(output/"weights.pt", map_location=device, weights_only=True))\n    dev_labels, dev_scores = predict(model, loaders["dev"], device)\n    chosen = choose_threshold(dev_labels, dev_scores)\n    # Held-out set is touched for prediction only after model/threshold selection is done.\n    labels, scores = predict(model, loaders["test"], device)\n    evaluation = metrics(labels, scores, chosen["threshold"])\n    test_rows = [r for r in rows if r["split"] == "test"]\n    (output/"test_predictions.json").write_text(json.dumps([\n        {"relative": r["relative"], "label": y, "score": s} for r,y,s in zip(test_rows, labels, scores)\n    ], indent=2), encoding="utf-8")\n    manifest = {"schema": 1, "architecture": "torchvision-r3d_18", "version": args.version,\n                "purpose": "test_only" if args.smoke else "research_trained", "labels": ["nonfight", "fight"],\n                "weights": "weights.pt", "sha256": sha256(output/"weights.pt"), "preprocess": PREPROCESS,\n                "threshold": chosen["threshold"], "evaluation": evaluation,\n                "training": {"dataset": "Synthetic plumbing test" if args.smoke else args.dataset_name,\n                             "seed": args.seed, "epochs": args.epochs, "pretrained": not args.smoke,\n                             "split_sha256": sha256(output/"split.json"), "grouped": bool(args.groups),\n                             "curation_sha256": sha256(output/"curation.json") if (output/"curation.json").is_file() else None,\n                             "counts": {s:len(loaders[s].dataset) for s in loaders}, "development": chosen,\n                             "seconds": round(time.perf_counter()-start, 3), "device": str(device),\n                             "hardware": torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU",\n                             "torch": torch.__version__, "torchvision": torchvision.__version__,\n                             "created_at": datetime.now(timezone.utc).isoformat()},\n                "limitations": ["Uncalibrated clip score, not misconduct probability.",\n                                "Not validated on Pakistani mobile/police interactions.",\n                                "Exact duplicate screening does not detect near-duplicate/source leakage.",\n                                "No weapon, identity, gesture, audio or bribery classification."]}\n    (output/"manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    shutil.make_archive(str(output), "zip", output)\n    print(f"Exported {output}.zip ({manifest[\'purpose\']}). Held-out metrics: {json.dumps(evaluation)}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--dataset", required=True)\n    parser.add_argument("--dataset-name", default="RWF-2000 (operator supplied)", help="Dataset name/source recorded in the exported manifest")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--groups", help="JSON mapping dataset-relative path to source/session group")\n    parser.add_argument("--version", default="rwf-r3d18-v1")\n    parser.add_argument("--epochs", type=int, default=8)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--workers", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--authorized-dataset", action="store_true")\n    parser.add_argument("--smoke", action="store_true", help="Random initialization, one training batch/epoch; exported bundle is rejected by server")\n    train(parser.parse_args())\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, source in FILES.items():
    target = SOURCE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
sys.path.insert(0, str(SOURCE))
for module_name in list(sys.modules):
    if module_name in ('training', 'sentinel_ml') or module_name.startswith(('training.', 'sentinel_ml.')):
        del sys.modules[module_name]
importlib.invalidate_caches()
import torch, torchvision, cv2, numpy
print('Torch:', torch.__version__, 'Torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
DATASET_ROOT = None  # Or the exact RWF-2000 folder containing train/ and val/
DATASET_NAME = 'RWF-2000 via magicearth25/video-violence-detection-dataset'
AUTHORIZED_DATASET = False
GROUPS_FILE = None  # Optional JSON: dataset-relative video path -> original source/session ID
EPOCHS = 8
BATCH_SIZE = 4  # Reduce to 2 or 1 if GPU memory is exhausted
VERSION = 'rwf-r3d18-v1'
OUTPUT = Path('/kaggle/working/sentinel-' + VERSION)
assert AUTHORIZED_DATASET, 'Confirm dataset authorization before running training.'
from training.data import find_root, dataset_roots
candidates = dataset_roots('/kaggle/input')
print('Available subsets:', [str(p) for p in candidates])
if DATASET_ROOT:
    dataset = find_root(DATASET_ROOT)
else:
    rwf = [p for p in candidates if p.name.casefold().replace('-', '').replace('_', '') == 'rwf2000']
    assert len(rwf) == 1, 'Set DATASET_ROOT to one RWF-2000 root. Do not merge collections or duplicate RWF copies.'
    dataset = rwf[0]
print('Dataset root:', dataset)
from training.curation import curate
import tempfile
dataset = curate(dataset, Path(tempfile.mkdtemp(prefix='sentinel-rwf-curated-', dir='/kaggle/working')), GROUPS_FILE, overlap_policy='keep_holdout')


In [ ]:
command = [sys.executable, '-m', 'training.train_violence', '--dataset', str(dataset),
           '--output', str(OUTPUT), '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
           '--version', VERSION, '--dataset-name', DATASET_NAME, '--authorized-dataset']
if GROUPS_FILE:
    command += ['--groups', GROUPS_FILE]
subprocess.run(command, cwd=SOURCE, check=True)


In [ ]:
manifest = json.loads((OUTPUT / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2))
from IPython.display import FileLink, display
display(FileLink(str(OUTPUT) + '.zip'))


## Bring the result into SentinelAI
Extract the ZIP into `.local/models/violence-v1`. Install `requirements-vision.txt` in the backend Python environment. Run `python -m backend.model_check .local/models/violence-v1`, set `SENTINEL_VIOLENCE_MODEL` to that folder and restart the backend.

Upload a **new**, authorized short recording. Previously processed originals were deleted for privacy, so they cannot be re-analyzed from the blurred copies. Inspect the processing record, timestamped events, model score, local alert and report. A model score is not a calibrated probability. A clip flag requests human review; it does not identify who acted or justify enforcement.

Keep the split manifest, history and test predictions with the weights for your FYP report. Near-duplicate/source leakage and camera/domain shift still need independent checks. Training may take longer than a Kaggle session; this baseline saves the best weights each epoch but does not implement optimizer-state resume. Preserve interrupted outputs and rerun with a new output/version.
